# Notebook 07: Neural Networks and Multi-Armed Bandits

## Session Overview
Practice a simple scikit-learn neural network and a reinforcement-learning style bandit simulation.

These TF notebooks are practice materials. They reinforce lecture notes and John V. Guttag, *Introduction to Computation and Programming Using Python*, Third Edition. They do not replace lecture or reading.

## Learning Goals
- Explain the basic structure of a neural network.
- Train a simple neural network using scikit-learn.
- Evaluate predictions.
- Explain the basic idea of reinforcement learning.
- Simulate a multi-armed bandit problem.
- Compare random, greedy, and epsilon-greedy strategies.

## Warm-up Questions
1. How might a computer recognize a handwritten digit?
2. What is the difference between exploring and exploiting?
3. Where do you see trial-and-error learning in real life?

## Part A: Neural Networks

## Section 1: What is a Neural Network?

A neural network takes inputs, passes them through hidden layers with weights and activations, produces outputs, measures loss, and updates weights during training.

## Section 2: Load Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
X = digits.data
y = digits.target

fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, image, label in zip(axes, digits.images[:5], y[:5]):
    ax.imshow(image, cmap="gray")
    ax.set_title(f"Label {label}")
    ax.axis("off")
plt.show()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

## Section 3: Train a Simple Neural Network

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(hidden_layer_sizes=(32,), max_iter=500, random_state=42)
mlp.fit(X_train, y_train)
mlp_predictions = mlp.predict(X_test)

## Section 4: Evaluate

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

mlp_accuracy = accuracy_score(y_test, mlp_predictions)
print("MLP accuracy:", mlp_accuracy)
print(confusion_matrix(y_test, mlp_predictions))

In [ ]:
correct_indices = np.where(mlp_predictions == y_test)[0][:3]
incorrect_indices = np.where(mlp_predictions != y_test)[0][:3]

fig, axes = plt.subplots(2, 3, figsize=(7, 4))
for ax, idx in zip(axes[0], correct_indices):
    ax.imshow(X_test[idx].reshape(8, 8), cmap="gray")
    ax.set_title(f"Correct: {mlp_predictions[idx]}")
    ax.axis("off")
for ax, idx in zip(axes[1], incorrect_indices):
    ax.imshow(X_test[idx].reshape(8, 8), cmap="gray")
    ax.set_title(f"Pred {mlp_predictions[idx]}, true {y_test[idx]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

### Exercise 4.1: Change Hidden Layer Size and Max Iterations

In [ ]:
experiment_results = []
# TODO: Try hidden layer sizes (16,), (32,), and (64,).
# TODO: Store the accuracy for each model.
pd.DataFrame(experiment_results)

## Section 5: Limitations

Neural networks can overfit, take time to train, be hard to interpret, and need enough data. TensorFlow and PyTorch are powerful tools, but this notebook uses scikit-learn so setup stays simple.

## Part B: Multi-Armed Bandits

## Section 6: Reinforcement Learning Intuition

Reinforcement learning involves an agent choosing actions and receiving rewards. The agent balances exploration and exploitation.

## Section 7: Build Bandit Environment

In [ ]:
reward_probabilities = [0.2, 0.5, 0.8]

def pull_arm(arm, rng):
    return 1 if rng.random() < reward_probabilities[arm] else 0

def run_random_strategy(rounds=100, seed=42):
    rng = np.random.default_rng(seed)
    rewards = []
    for _ in range(rounds):
        arm = rng.integers(0, len(reward_probabilities))
        rewards.append(pull_arm(arm, rng))
    return np.array(rewards)

random_rewards = run_random_strategy(100)
print("Total random reward:", random_rewards.sum())

## Section 8: Epsilon-Greedy Strategy

In [ ]:
def run_epsilon_greedy(rounds=1000, epsilon=0.1, seed=42):
    rng = np.random.default_rng(seed)
    counts = np.zeros(len(reward_probabilities))
    estimates = np.zeros(len(reward_probabilities))
    rewards = []

    for _ in range(rounds):
        if rng.random() < epsilon:
            arm = rng.integers(0, len(reward_probabilities))
        else:
            arm = int(np.argmax(estimates))

        reward = pull_arm(arm, rng)
        rewards.append(reward)
        counts[arm] += 1
        estimates[arm] += (reward - estimates[arm]) / counts[arm]

    return np.array(rewards), estimates, counts

rewards, estimates, counts = run_epsilon_greedy(rounds=1000, epsilon=0.1)
print("Total reward:", rewards.sum())
print("Estimates:", estimates)
print("Counts:", counts)

### Exercise 8.1: Compare Rounds and Epsilon Values

In [ ]:
bandit_results = []
# TODO: Run for 100, 500, and 1000 rounds.
# TODO: Compare epsilon values 0.0, 0.1, and 0.3.
pd.DataFrame(bandit_results)

In [ ]:
plt.figure(figsize=(7, 4))
for epsilon in [0.0, 0.1, 0.3]:
    rewards, _, _ = run_epsilon_greedy(rounds=500, epsilon=epsilon, seed=42)
    plt.plot(np.cumsum(rewards), label=f"epsilon={epsilon}")
plt.title("Cumulative Reward for Epsilon-Greedy")
plt.xlabel("Round")
plt.ylabel("Cumulative Reward")
plt.legend()
plt.show()

## Section 9: Compare Strategies

In [ ]:
def run_greedy_strategy(rounds=1000, seed=42):
    return run_epsilon_greedy(rounds=rounds, epsilon=0.0, seed=seed)[0]

comparison = pd.DataFrame([
    {"strategy": "random", "total_reward": run_random_strategy(1000, seed=42).sum()},
    {"strategy": "greedy", "total_reward": run_greedy_strategy(1000, seed=42).sum()},
    {"strategy": "epsilon-greedy", "total_reward": run_epsilon_greedy(1000, epsilon=0.1, seed=42)[0].sum()},
])
comparison

### Exercise 9.1: Which Strategy Performs Best?

In [ ]:
strategy_sentence = ""
# TODO: Which strategy performs best? Why does exploration help?
print(strategy_sentence)

## Section 10: Challenge

In [ ]:
# TODO: Add a fourth arm and rerun the simulation.
pass

## Section 11: Reflection

## Reflection Questions
1. How is reinforcement learning different from supervised learning?
2. What does it mean to learn from reward?
3. Where might bandit algorithms appear in real life?

## Optional Extension for Advanced Students
Try changing reward probabilities over time to simulate a non-stationary environment.